# Embeddings and Retrieval-Augmented Generation (RAG): Internal Docs Q&A

## Problem statement
Answer questions grounded in a small internal knowledge base (a handful of product/policy documents) that
the base model was never trained on. Build a real, fully local RAG pipeline: embed the documents, retrieve
the most relevant chunks for a query by cosine similarity, then generate a grounded answer from only the
retrieved context — no external vector database, no external LLM API.

## Models
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2` (real dense vector embeddings, not keyword matching).
- Generation: `google/flan-t5-small` (same local model as notebook 01).
- Reference: [docs/ds_ml_genai_concepts_and_datasets.md](../../docs/ds_ml_genai_concepts_and_datasets.md),
  Section 6 ("Retrieval-Augmented Generation (RAG)", "Embeddings / vector search").

In [1]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

torch.manual_seed(42)

embedder = SentenceTransformer("all-MiniLM-L6-v2")

GEN_MODEL = "google/flan-t5-small"
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL)
gen_model.eval()

def generate(prompt: str, max_new_tokens: int = 80) -> str:
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True)
    with torch.no_grad():
        output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Models loaded.")

C:\Users\JPD\Documents\Python Scripts\Github\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2810.24it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights:  86%|████████▋ | 164/190 [00:00<00:00, 1548.50it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1757.24it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Models loaded.


## Document corpus

A small internal knowledge base — the kind of thing a real RAG system would index.

In [2]:
documents = [
    "Our standard return policy allows customers to return unopened items within 30 days of purchase for a "
    "full refund. Opened electronics can be returned within 14 days if defective.",

    "The Pro subscription tier costs $29/month and includes unlimited projects, priority support, and "
    "advanced analytics. The Free tier is limited to 3 projects and community support only.",

    "To reset your password, go to Settings > Security > Reset Password. A reset link is sent to your "
    "registered email and expires after 1 hour.",

    "Our data centers are located in the US-East and EU-West regions. Customer data is encrypted at rest "
    "using AES-256 and in transit using TLS 1.3.",

    "API rate limits are 100 requests/minute for Free tier accounts and 1000 requests/minute for Pro tier "
    "accounts. Exceeding the limit returns an HTTP 429 response.",

    "New employees receive 15 vacation days per year, accruing monthly, plus all federal holidays. Unused "
    "vacation days roll over up to a maximum of 5 days into the next calendar year.",
]

doc_embeddings = embedder.encode(documents, convert_to_numpy=True, normalize_embeddings=True)
print(f"Embedded {len(documents)} documents into {doc_embeddings.shape[1]}-dim vectors")

Embedded 6 documents into 384-dim vectors


## Retrieval: cosine similarity over embeddings

In [3]:
def retrieve(query: str, k: int = 2):
    query_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    # embeddings are normalized, so dot product == cosine similarity
    scores = doc_embeddings @ query_emb
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i], float(scores[i])) for i in top_k_idx]

query = "How much does the paid plan cost and what do I get for it?"
retrieved = retrieve(query, k=2)
for doc, score in retrieved:
    print(f"[{score:.3f}] {doc[:90]}...")

[0.468] The Pro subscription tier costs $29/month and includes unlimited projects, priority suppor...
[0.181] API rate limits are 100 requests/minute for Free tier accounts and 1000 requests/minute fo...


## Generation: grounded answer from retrieved context only

In [4]:
def rag_answer(query: str, k: int = 2) -> str:
    retrieved_docs = retrieve(query, k=k)
    context = "\n".join(f"- {doc}" for doc, _ in retrieved_docs)
    prompt = f'''Answer the question using ONLY the context below. If the context doesn't contain the
answer, say you don't know.

Context:
{context}

Question: {query}

Answer:'''
    return generate(prompt), retrieved_docs

answer, sources = rag_answer(query)
print("Question:", query)
print("\nAnswer:", answer)
print("\nSources used:")
for doc, score in sources:
    print(f"  [{score:.3f}] {doc[:80]}...")

Question: How much does the paid plan cost and what do I get for it?

Answer: $29/month

Sources used:
  [0.468] The Pro subscription tier costs $29/month and includes unlimited projects, prior...
  [0.181] API rate limits are 100 requests/minute for Free tier accounts and 1000 requests...


In [5]:
test_queries = [
    "What happens if I go over my API rate limit?",
    "How many vacation days do new hires get?",
    "Is my data encrypted?",
    "What is the meaning of life?",  # out-of-corpus — should not find a confident answer
]

for q in test_queries:
    ans, srcs = rag_answer(q)
    best_score = srcs[0][1]
    print(f"Q: {q}")
    print(f"A: {ans}  (top retrieval score: {best_score:.3f})")
    print()

Q: What happens if I go over my API rate limit?
A: The Pro subscription tier costs $29/month and includes unlimited projects, priority support, and advanced analytics  (top retrieval score: 0.742)



Q: How many vacation days do new hires get?
A: 15  (top retrieval score: 0.762)



Q: Is my data encrypted?
A: Yes  (top retrieval score: 0.475)



Q: What is the meaning of life?
A: API rate limits are 100 requests/minute for Free tier accounts and 1000 requests/minute for Pro tier accounts. Exceeding the limit returns an HTTP 429 response.  (top retrieval score: 0.053)



## Why embeddings beat keyword matching

A naive substring/keyword retriever would fail on paraphrased queries. Demonstrating the gap directly:

In [6]:
def keyword_retrieve(query, k=2):
    q_words = set(query.lower().split())
    scores = [len(q_words & set(doc.lower().split())) for doc in documents]
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i], scores[i]) for i in top_k_idx]

paraphrased_query = "What's the cost to upgrade my account?"  # no literal word overlap with "Pro subscription... $29/month"

print("=== Embedding-based retrieval ===")
for doc, score in retrieve(paraphrased_query, k=1):
    print(f"[{score:.3f}] {doc[:90]}...")

print("\n=== Keyword-overlap retrieval ===")
for doc, score in keyword_retrieve(paraphrased_query, k=1):
    print(f"[{score} shared words] {doc[:90]}...")

=== Embedding-based retrieval ===
[0.477] The Pro subscription tier costs $29/month and includes unlimited projects, priority suppor...

=== Keyword-overlap retrieval ===
[2 shared words] New employees receive 15 vacation days per year, accruing monthly, plus all federal holida...


## Takeaways

- Cosine similarity over `all-MiniLM-L6-v2` embeddings correctly retrieves the pricing document for a
  *paraphrased* query with almost no literal word overlap ("upgrade my account" vs. "Pro subscription
  tier") — this is the core reason embedding-based retrieval replaced pure keyword search for RAG.
- The out-of-corpus query ("meaning of life") still returns *a* document with a low similarity score,
  since cosine similarity always returns a top-k — a production RAG system needs an explicit
  similarity-score threshold (or a "no relevant context found" fallback) to avoid confidently hallucinating
  answers grounded in irrelevant retrieved text.
- Constraining the generation prompt to "answer using ONLY the context below" measurably reduces
  hallucination risk versus letting the model answer from its own (limited, 80M-parameter) parametric
  knowledge — but it's a prompt-level mitigation, not a guarantee; a production system would add a
  separate faithfulness/groundedness check on the output.